### Uploading a Folder to Colab

There are a few ways to upload a folder:

1.  **Drag and Drop:** The easiest way is to drag your folder from your local machine directly into the Colab file browser (the folder icon on the left sidebar). It will typically be placed in `/content/`.
2.  **Using `files.upload()`:** For programmatic upload, you can use `google.colab.files.upload()`. However, this uploads individual files, so it's less convenient for entire folders.
3.  **Mount Google Drive:** If your scripts are in Google Drive, you can mount your Drive and then navigate to the folder. This is often the most robust solution for larger projects or persistent storage.

In [ ]:
from google.colab import drive

# Mount your Google Drive
drive.mount('/content/drive')

# You can then navigate to your folder, for example:
%cd /content/drive/MyDrive/nba_bets

Mounted at /content/drive
/content/drive/MyDrive/nba_bets


In [ ]:
import sys
import os

# Define the path to your scripts folder
scripts_folder_path = '/content/drive/MyDrive/nba_bets'

# Check if the folder exists (optional, but good practice)
if not os.path.exists(scripts_folder_path):
    print(f"Warning: The folder '{scripts_folder_path}' does not exist. Please upload it first.")
else:
    # Add the folder to the Python path
    if scripts_folder_path not in sys.path:
        sys.path.insert(0, scripts_folder_path)
        print(f"Added '{scripts_folder_path}' to sys.path")
    else:
        print(f"'{scripts_folder_path}' is already in sys.path")

Added '/content/drive/MyDrive/nba_bets' to sys.path


In [ ]:
# Navigate to the directory containing pyproject.toml
# Assuming scripts_folder_path is already defined and points to '/content/drive/MyDrive/nba_bets'
%cd {scripts_folder_path}

/content/drive/MyDrive/nba_bets


In [ ]:
# Uninstall the current package to ensure a clean install with optional dependencies
!pip uninstall -y nba-bets

# Reinstall with the 'gpu' optional dependency
!pip install .[gpu]

Processing /content/drive/MyDrive/nba_bets
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 146.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os

from pathlib import Path
from typing import Optional

from src.config.paths import DEFAULT_TRAIN_LLM_CONFIG_PATH
from src.ml.config.loader import load_llm_training_config
from src.ml.tracking.mlflow_tracker import MLflowTracker

from src.ml.llm.finetune import (
    preflight,
    resolve_run_name,
    run_training,
    get_hf_token,
    build_lora_config,
    build_hub_model_id,
    find_resumable_checkpoint,
    build_tokenizer,
    build_base_model,
    build_sft_config,
    load_train_dataset,
    load_eval_dataset,
    resolve_max_length
)

from src.utils.logging_config import get_logger

from src.ml.config.schema import LLMTrainingConfig
from src.utils.logging_config import get_logger
from src.config.secrets import (
    GLOBAL_SECRETS_FILE,
    HF_TOKEN_ENV_VAR,
    WANDB_TOKEN_ENV_VAR,
    require_env,
)

from src.ml.llm.train import _configure_wandb_env


from google.colab import userdata

logger = get_logger(__name__)

In [ ]:
## Training ###
from huggingface_hub import login
from trl import SFTTrainer

In [ ]:
LLM_CONFIG = "llama31_8b_qlora"
config_path = f"configs/train_llm/{LLM_CONFIG}.yaml"
# max_train_samples = 300
run_name = "test_run2"

In [ ]:
resolved = Path(config_path) if config_path else DEFAULT_TRAIN_LLM_CONFIG_PATH
if not resolved.exists():
    raise FileNotFoundError(
        f"LLM training config not found: {resolved}. "
        "Pass config_path= or --config with a valid YAML "
        "(e.g. configs/train_llm/llama31_8b_qlora.yaml)."
    )
config = load_llm_training_config(resolved)

In [ ]:
if max_train_samples is not None:
    config.data.max_train_samples = max_train_samples

resolved_run_name = resolve_run_name(config, run_name)

In [ ]:
preflight(config)

In [ ]:
_configure_wandb_env(config)
mlflow_config = config.tracking.mlflow

mlflow_config

LLMMLflowConfig(enabled=True, experiment_name='nba_bets_llm', tracking_uri='sqlite:///mlflow.db')

In [ ]:
token = get_hf_token()
login(token, add_to_git_credential=True)

hub_model_id = build_hub_model_id(config, run_name)
output_dir = Path(config.paths.output_dir) / run_name

In [ ]:
resume = "auto"

In [ ]:
checkpoint = None
if resume == "auto":
    checkpoint = find_resumable_checkpoint(hub_model_id, output_dir, token)
else:
    print("resume=never - starting from scratch.")

In [ ]:
print(f"Run name:     {run_name}")
print(f"Hub model id: {hub_model_id}")
print(f"Output dir:   {output_dir}")
print(f"Resuming from: {checkpoint or 'scratch'}")

Run name:     test_run2
Hub model id: fformenti/test_run2
Output dir:   outputs/llm/test_run2
Resuming from: scratch


In [ ]:
print(f"Checking for resumable checkpoint under model ID '{hub_model_id}' or local output directory '{output_dir}'...")
checkpoint = find_resumable_checkpoint(hub_model_id, output_dir, token)
if checkpoint:
    print(f"✅ Resumable checkpoint found: {checkpoint}")
else:
    print("❌ No checkpoint found on Hugging Face Hub or locally. It will start from scratch.")

In [ ]:
train = load_train_dataset(config)

README.md:   0%|          | 0.00/647 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.03MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  764kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  269kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/23247 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2949 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1018 [00:00<?, ? examples/s]

In [ ]:
train

Dataset({
    features: ['game_id', 'prompt', 'completion', 'margin', 'margin_scale'],
    num_rows: 23247
})

In [ ]:
evaluation = load_eval_dataset(config) if config.training.eval_strategy != "no" else None

In [ ]:
VAL_SIZE = 800
evaluation = evaluation_full.select(range(VAL_SIZE))

In [ ]:
tokenizer = build_tokenizer(config)

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

In [ ]:
base_model = build_base_model(config, tokenizer)

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [ ]:
peft_config=build_lora_config(config)

In [ ]:
peft_config

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=32, target_modules={'k_proj', 'q_proj', 'o_proj', 'v_proj'}, exclude_modules=None, lora_alpha=64, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)

In [ ]:
from src.ml.llm.finetune import (
    _report_to
)

def build_sft_config(
    config: LLMTrainingConfig, run_name: str, output_dir: Path, max_length: int
):
    """Map the YAML config onto a trl ``SFTConfig``."""
    from trl import SFTConfig

    training = config.training
    return SFTConfig(
        output_dir=str(output_dir),
        run_name=run_name,
        num_train_epochs=training.epochs,
        per_device_train_batch_size=training.per_device_train_batch_size,
        per_device_eval_batch_size=training.per_device_eval_batch_size,
        eval_strategy=training.eval_strategy,
        eval_steps=training.eval_steps,
        gradient_accumulation_steps=training.gradient_accumulation_steps,
        gradient_checkpointing=training.gradient_checkpointing,
        optim=training.optimizer,
        learning_rate=training.learning_rate,
        lr_scheduler_type=training.lr_scheduler_type,
        warmup_steps=training.warmup_ratio,
        weight_decay=training.weight_decay,
        max_grad_norm=training.max_grad_norm,
        max_steps=-1,
        bf16=training.bf16,
        fp16=training.fp16,
        seed=training.seed,
        logging_steps=training.logging_steps,
        disable_tqdm=False,
        save_strategy="steps",
        save_steps=training.save_steps,
        save_total_limit=training.save_total_limit,
        report_to=_report_to(config),
        completion_only_loss=training.train_on_completion_only,
        max_length=max_length,
        push_to_hub=config.hub.push_checkpoints,
        hub_strategy="every_save",
        hub_model_id=build_hub_model_id(config, run_name),
        hub_private_repo=config.hub.private,
    )

In [ ]:
max_length = resolve_max_length(config, tokenizer)

In [ ]:
max_length

464

In [ ]:
# max_length = 1024 * 16
train_args = build_sft_config(config, run_name, output_dir, max_length)

In [ ]:
train_args

SFTConfig(output_dir='outputs/llm/test_run2', per_device_train_batch_size=2, num_train_epochs=2, max_steps=-1, learning_rate=0.0001, lr_scheduler_type=<SchedulerType.COSINE: 'cosine'>, lr_scheduler_kwargs=None, warmup_steps=0.03, optim=<OptimizerNames.PAGED_ADAMW: 'paged_adamw_32bit'>, optim_args=None, weight_decay=0.001, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, optim_target_modules=None, gradient_accumulation_steps=4, average_tokens_across_devices=True, max_grad_norm=0.3, label_smoothing_factor=0.0, bf16=True, fp16=False, bf16_full_eval=False, fp16_full_eval=False, tf32=None, gradient_checkpointing=True, gradient_checkpointing_kwargs=None, torch_compile=False, torch_compile_backend=None, torch_compile_mode=None, use_liger_kernel=False, liger_kernel_config=None, use_cache=False, neftune_noise_alpha=None, torch_empty_cache_steps=None, auto_find_batch_size=False, logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_steps=50, logging_first_step=False, log_on_each_node=

In [ ]:
trainer = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=evaluation,
    peft_config=peft_config,
    args=train_args,
)

Adding EOS to train dataset:   0%|          | 0/23247 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/23247 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/23247 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/23247 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/23247 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

In [ ]:
import logging
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)
result = trainer.train(resume_from_checkpoint=str(checkpoint) if checkpoint else None)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: felipeformenti to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


2026/10/16 04:18:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/16 04:18:11 INFO mlflow.store.db.utils: Updating database tables


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
200,1.347173,1.321773,1.330953,0.530417,661603.000000
400,1.314135,1.379724,1.298879,0.553750,1321818.000000
600,1.319929,1.344868,1.224639,0.574167,1987271.000000
800,1.305620,1.308970,1.278671,0.554583,2649422.000000
1000,1.298370,1.302239,1.296433,0.561250,3309441.000000
1200,1.309461,1.305727,1.337160,0.554583,3971759.000000
1400,1.309598,1.311535,1.295207,0.540417,4634427.000000
1600,1.292034,1.322630,1.301739,0.532083,5295471.000000
1800,1.290270,1.308334,1.270941,0.556250,5958137.000000
2000,1.282104,1.304940,1.246564,0.565417,6621883.000000


wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-200)... Done. 0.8s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-400)... Done. 0.7s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-600)... Done. 0.8s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-800)... Done. 0.8s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-1000)... Done. 0.7s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-1200)... Done. 0.7s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-1400)... Done. 0.8s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-1600)... Done. 0.7s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-1800)... Done. 0.8s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-2000)... Done. 0.8s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-2200)... Done. 0.7s
wandb: Adding 

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
200,1.347173,1.321773,1.330953,0.530417,661603.000000
400,1.314135,1.379724,1.298879,0.553750,1321818.000000
600,1.319929,1.344868,1.224639,0.574167,1987271.000000
800,1.305620,1.308970,1.278671,0.554583,2649422.000000
1000,1.298370,1.302239,1.296433,0.561250,3309441.000000
1200,1.309461,1.305727,1.337160,0.554583,3971759.000000
1400,1.309598,1.311535,1.295207,0.540417,4634427.000000
1600,1.292034,1.322630,1.301739,0.532083,5295471.000000
1800,1.290270,1.308334,1.270941,0.556250,5958137.000000
2000,1.282104,1.304940,1.246564,0.565417,6621883.000000


wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-4200)... Done. 0.8s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-4400)... Done. 0.7s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-4600)... Done. 0.7s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-4800)... Done. 0.8s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-5000)... Done. 0.8s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-5200)... Done. 0.8s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-5400)... Done. 0.7s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-5600)... Done. 0.7s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-5800)... Done. 0.7s
wandb: Adding directory to artifact (outputs/llm/test_run2/checkpoint-5812)... Done. 0.8s


In [ ]:
trainer.model.push_to_hub(hub_model_id, private=config.hub.private)
logger.info(f"Pushed final adapter to the hub: {hub_model_id}")

README.md:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  29%|##9       | 16.0MB / 54.6MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


In [ ]:
result.metrics

{'train_runtime': 34814.8625,
 'train_samples_per_second': 1.335,
 'train_steps_per_second': 0.167,
 'total_flos': 9.059158222415954e+17,
 'train_loss': 1.31354375026674,
 'epoch': 2.0}

In [ ]:
from src.ml.llm.finetune import (
    build_trainer_class
)

In [ ]:
trainer = build_trainer_class()(
    model=base_model,
    train_dataset=train,
    eval_dataset=evaluation,
    peft_config=build_lora_config(config),
    args=build_sft_config(config, run_name, output_dir, max_length),
)

In [ ]:
from src.ml.llm.evaluate import (
    run_evaluation,
    run_tracked_evaluation,
    build_predictor,
    # build_batch_predictor
)

In [ ]:
config

LLMTrainingConfig(description='QLoRA SFT of Llama-3.1-8B on nba-bets point-differential prompts', run_name=None, base_model='meta-llama/Meta-Llama-3.1-8B', hub=HubConfig(user='fformenti', project_name='nba-bets', private=True, push_checkpoints=True), data=LLMDataConfig(dataset_name='fformenti/nba-bets', train_split='train', eval_split='validation', prompt_field='prompt', max_sequence_length=1024, max_train_samples=300, max_eval_samples=None, source_experiment_config='configs/train/llm_features.yaml', serialization_format='labeled'), quantization=QuantizationConfig(mode='8bit', compute_dtype='bfloat16', bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True), lora=LoraSettings(r=32, alpha=64, dropout=0.1, target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'], bias='none'), training=LLMTrainerConfig(epochs=2, per_device_train_batch_size=2, gradient_accumulation_steps=4, learning_rate=0.0001, lr_scheduler_type='cosine', warmup_ratio=0.03, optimizer='paged_adamw_32bit', weight_decay=0

In [ ]:
split = "test"
size = 1000
revision = None

In [ ]:
hub_model_id

'fformenti/test_run'

In [ ]:
def load_eval_dataset(config: LLMTrainingConfig, split: str):
    from datasets import load_dataset

    # dataset = load_dataset(config.data.dataset_name, token=get_hf_token())
    token = userdata.get('HF_TOKEN')
    dataset = load_dataset(config.data.dataset_name, token=token)
    if split not in dataset:
        raise KeyError(
            f"Split '{split}' not in {config.data.dataset_name}. "
            f"Available: {sorted(dataset.keys())}"
        )
    return dataset[split]

In [ ]:
data = load_eval_dataset(config, split)


In [ ]:
first = data[0]

In [ ]:
# predictor = build_predictor(config, hub_model_id, revision=revision)

In [ ]:
predictor(first)

'+13<|end_of_text|>'

In [ ]:
config.evaluation.max_new_tokens

8

In [ ]:
def build_batch_predictor(config: LLMTrainingConfig, hub_model_id: str, revision=None):
    """Return a ``predict_batch(datapoints) -> list[str]`` that decodes them together.

    Prompts are left-padded so every row's prompt ends at the same column: the
    generated tokens then start at one shared offset, and decoding is greedy so
    the answers match :func:`build_predictor` row for row.
    """
    import torch

    tokenizer, model = _load_adapter_model(config, hub_model_id, revision=revision)
    # Decoder-only generation needs left padding; training keeps right padding.
    tokenizer.padding_side = "left"
    max_new_tokens = config.evaluation.max_new_tokens

    def predict_batch(items) -> list[str]:
        prompts = [item[config.data.prompt_field] for item in items]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        prompt_len = inputs["input_ids"].shape[1]
        return tokenizer.batch_decode(output_ids[:, prompt_len:], skip_special_tokens=True)

    return predict_batch

In [ ]:
from src.ml.llm.testers import ground_truth, DEFAULT_SIZE, ClassificationTester
from IPython.display import clear_output

class BatchedClassificationTester(ClassificationTester):
    """:class:`ClassificationTester` that scores ``batch_size`` rows per call.

    ``batch_predictor(list[datapoint]) -> list[str]`` answers a whole chunk at
    once, so a GPU model decodes the batch in one ``generate`` instead of one
    prompt at a time. Grading and charts are the parent's, unchanged.
    """

    def __init__(
        self,
        batch_predictor,
        data,
        title=None,
        size=DEFAULT_SIZE,
        batch_size=16,
        output_dir=None,
    ):
        super().__init__(batch_predictor, data, title=title, size=size)
        self.batch_size = batch_size
        self.output_dir = Path(output_dir) if output_dir else None

    def run_batch(self, start: int) -> None:
        rows = [self.data[i] for i in range(start, min(start + self.batch_size, self.size))]
        values = self.predictor(rows)
        if len(values) != len(rows):
            raise ValueError(f"batch_predictor returned {len(values)} outputs for {len(rows)} rows")
        for datapoint, value in zip(rows, values):
            guess = self.post_process(value)
            truth = ground_truth(datapoint)
            correct = self.is_correct(guess, truth)
            self.titles.append(f"game {datapoint['game_id']}")
            self.guesses.append(guess)
            self.truths.append(truth)
            self.corrects.append(correct)
            self.colors.append(self.color_for(correct))

    def metrics(self) -> dict:
        n_correct = sum(self.corrects)
        return {"accuracy": n_correct / self.size, "n_correct": n_correct, "size": self.size}

    def report(self):
        m = self.metrics()
        title = (
            f"{self.title} results<br><b>Accuracy:</b> {m['accuracy']:.1%} "
            f"({m['n_correct']}/{m['size']})"
        )
        figures = {
            "accuracy_trend": self.accuracy_trend_chart(),
            "confidence_accuracy": self.confidence_accuracy_chart(),
            "scatter": self.chart(title),
        }
        if self.output_dir:
            self.output_dir.mkdir(parents=True, exist_ok=True)
        for name, fig in figures.items():
            fig.show()
            if self.output_dir:
                fig.write_html(self.output_dir / f"{name}.html")

    def run(self) -> dict:
        self.size = min(self.size, len(self.data))
        with tqdm(total=self.size) as bar:
            for start in range(0, self.size, self.batch_size):
                self.run_batch(start)
                bar.update(len(self.corrects) - bar.n)
                bar.set_postfix(acc=f"{sum(self.corrects) / len(self.corrects):.1%}")
        clear_output(wait=True)
        self.report()
        return self.metrics()

In [ ]:
from transformers import set_seed

set_seed(config.evaluation.seed)

In [ ]:
def _load_adapter_model(config: LLMTrainingConfig, hub_model_id: str, revision=None):
    """Load the tokenizer and the base model with the fine-tuned adapter applied."""
    from peft import PeftModel

    tokenizer = build_tokenizer(config)
    base_model = build_base_model(config, tokenizer)
    model = PeftModel.from_pretrained(base_model, hub_model_id, revision=revision)
    model.eval()
    return tokenizer, model

In [ ]:
import torch; print([x/1e9 for x in torch.cuda.mem_get_info()])

[23.455727616, 23.65915136]


In [ ]:
from src.config.paths import DEFAULT_TRAIN_LLM_CONFIG_PATH, LLM_EVAL_OUTPUTS_DIR
output_dir = LLM_EVAL_OUTPUTS_DIR / run_name

predict_batch = build_batch_predictor(config, hub_model_id, revision=revision)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
from tqdm import tqdm

In [ ]:
data.num_rows

1018

In [ ]:
# metrics = ClassificationTester(
#     predictor,
#     data,
#     title=run_name,
#     size=size,
#     # output_dir=output_dir,
# ).run()

In [ ]:


metrics = BatchedClassificationTester(
    predict_batch,
    data,
    title=run_name,
    size=data.num_rows,
    batch_size=32,
    # output_dir=output_dir,
).run()

AttributeError: 'NoneType' object has no attribute 'show'